In [1]:
import pandas as pd, numpy as np, glob

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

files = glob.glob("../data/raw/*")
print(files)

path = [f for f in files if f.lower().endswith((".csv", ".xlsx"))][0]

if path.lower().endswith(".csv"):
    df = pd.read_csv(path, encoding="latin-1")
else:
    df = pd.concat(pd.read_excel(path, sheet_name=None).values(), ignore_index=True)

df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
print(df.columns.tolist())
df.head()


['../data/raw\\online_retail_II.csv']
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [2]:
print("Rows, columns:", df.shape)
df.info()
df.describe(include="all").T


Rows, columns: (1067371, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1067371 non-null  str           
 1   stockcode    1067371 non-null  str           
 2   description  1062989 non-null  str           
 3   quantity     1067371 non-null  int64         
 4   invoicedate  1067371 non-null  datetime64[us]
 5   price        1067371 non-null  float64       
 6   customer_id  824364 non-null   float64       
 7   country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 65.1 MB


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
invoice,1067371,53628,537434,1350,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stockcode,1067371,5305,85123A,5829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,"1,067,371.00",NaN,NaN,NaN,9.94,"-80,995.00",1.00,3.00,10.00,"80,995.00",172.71
invoicedate,1067371,NaN,NaN,NaN,2011-01-02 21:13:55.394029,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
price,"1,067,371.00",NaN,NaN,NaN,4.65,"-53,594.36",1.25,2.10,4.15,"38,970.00",123.55
customer_id,"824,364.00",NaN,NaN,NaN,"15,324.64","12,346.00","13,975.00","15,255.00","16,797.00","18,287.00","1,697.46"
country,1067371,43,United Kingdom,981330,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
n = len(df)
inv = df["invoice"].astype(str)

quality = pd.DataFrame({
    "check": [
        "Missing Customer ID",
        "Missing Description",
        "Duplicate rows",
        "Cancellation rows (invoice starts with C)",
        "Negative quantity rows",
        "Zero or negative price rows",
        "Non-product stock codes (postage, fees, samples...)",
    ],
    "rows": [
        df["customer_id"].isna().sum(),
        df["description"].isna().sum(),
        df.duplicated().sum(),
        inv.str.startswith("C").sum(),
        (df["quantity"] < 0).sum(),
        (df["price"] <= 0).sum(),
        (~df["stockcode"].astype(str).str.match(r"^\d{5}")).sum(),
    ],
})
quality["percent_of_rows"] = (quality["rows"] / n * 100).round(2)
quality


,check,rows,percent_of_rows
0,Missing Customer ID,243007,22.77
1,Missing Description,4382,0.41
2,Duplicate rows,34335,3.22
3,Cancellation rows (invoice starts with C),19494,1.83
4,Negative quantity rows,22950,2.15
5,Zero or negative price rows,6207,0.58
6,"Non-product stock codes (postage, fees, sample...",6093,0.57


In [4]:
nonprod = df[~df["stockcode"].astype(str).str.match(r"^\d{5}")]
print(nonprod["stockcode"].value_counts().head(15))

neg = df[df["quantity"] < 0]
print("Negative qty rows that are cancellations:",
      neg["invoice"].astype(str).str.startswith("C").sum(), "of", len(neg))

print(df[df["price"] <= 0]["description"].value_counts().head(10))


stockcode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64
Negative qty rows that are cancellations: 19493 of 22950
description
check                    162
?                         92
damages                   84
damaged                   81
found                     28
missing                   27
sold as set on dotcom     20
Damaged                   17
adjustment                16
OWL DOORSTOP              15
Name: count, dtype: int64


In [5]:
print("Date range:", df["invoicedate"].min(), "to", df["invoicedate"].max())
print("Countries:", df["country"].nunique())
print(df["country"].value_counts().head(10))

df["revenue"] = df["quantity"] * df["price"]
print("Customers (with ID):", df["customer_id"].nunique())
print("Invoices:", df["invoice"].nunique())
print("Gross revenue (before cleaning): £{:,.0f}".format(df["revenue"].sum()))


Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Countries: 43
country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64
Customers (with ID): 5942
Invoices: 53628
Gross revenue (before cleaning): £19,287,251


In [6]:
import os
os.makedirs("../outputs/reports", exist_ok=True)
quality.to_csv("../outputs/reports/data_quality_summary.csv", index=False)
